In [1]:
import sys
sys.path.insert(0, '../lib')

import gc

import scanpy as sc
import pandas as pd
import numpy as np

import common_data
import common_plots

/projects/b1196/envs/serniczek/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Exporting a downsampled object for preprint cellbrowser

10% downsample, anonymize

Metadata:
- cell_id -> replace cell_id
- individual -> anonymize, rename “Sample”
- n_genes_by_counts -> “Number of genes”
- total_counts -> “Number of UMIs”
- pct_counts_mito -> “% of mitochondrial reads”
- pct_counts_ribo -> “% of ribosomal reads”
- Level_6 -> “Cell type”
- Patient id -> “Patient”
- SOFA score
- day_of_hospitalization -> log2
- days_on_ventilator -> log2
- Sample group
- Pneumonia episode type
- Mortality
- Discharge_disposition
- Immunocompromised
- Sex

Following the recommendation from Cellbrowser to use just raw counts for visualization of expression

In [2]:
ds = sc.read_h5ad(common_data.SC_RAW)

In [3]:
rng = np.random.default_rng(74451)

We excluded these 2 samples, but they are still in `raw` object

In [ ]:
all_cells = ds.obs_names
cells = rng.choice(all_cells, size=len(all_cells) // 10, replace=False)

In [5]:
ds_small = ds[cells].copy()

In [6]:
del ds

In [7]:
gc.collect()

2470

In [8]:
ds_small.shape

(243867, 19488)

Transfer UMAP

In [9]:
ds_processed = sc.read_h5ad(common_data.SC_NORM)

In [10]:
ds_small.obs_names.isin(ds_processed.obs_names).all()

True

In [11]:
ds_small.obsm['X_umap'] = ds_processed[ds_small.obs_names].obsm['X_umap'].copy()

In [12]:
del ds_processed

In [13]:
gc.collect()

2743

Build our necessary metadata

In [14]:
sc_labels = pd.read_csv(common_data.SC_LABELS, index_col=0)

In [ ]:
ds_small.obs = ds_small.obs[[
    'library_id', 'individual', 'n_genes_by_counts', 'total_counts',
    'pct_counts_mito', 'pct_counts_ribo', 'Level_6', 'Sex_genes'
]].copy()

In [16]:
RARE_PATHOGENS = [
    'Gram-*; SARS-CoV-2',
    'Gram-*; SARS-CoV-2; Gram+',
    'Other viruses',
    'Gram-*; Pseudomonas aeruginosa',
    'Pseudomonas aeruginosa; Gram+',
    'Other viruses; Pseudomonas aeruginosa',
    'Gram-*; Other viruses; SARS-CoV-2'
]

In [17]:
METADATA_FIELDS = {
    'bal_barcode': 'Sample ID',
    'perturbation_groups_2': 'Pathogen groups',
    'pathogen_groups': 'pathogen_groups',
    'SOFA_score': 'SOFA score',
    'days_on_ventilator': 'Days on ventilator (log)',
    'day_of_hospitalization': 'Day of hospitalization (log)',
    'episode_type': 'Pneumonia episode type',
    'Binary_outcome': 'Mortality',
    'Discharge_disposition': 'Discharge disposition',
    'Immunocompromised_flag': 'Immunocompromised',
    'is_culture_negative_pneumonia': 'Pathogen negative',
    'cohort': 'Cohort'
}

In [18]:
sc_labels = sc_labels[METADATA_FIELDS.keys()].set_index('bal_barcode')

In [19]:
sc_labels = sc_labels.rename(columns=METADATA_FIELDS)

In [20]:
sc_labels.loc[sc_labels['Pathogen negative'].fillna(False), 'Sample group'] = 'Pathogen-negative pneumonia'

In [21]:
sc_labels.loc[sc_labels['pathogen_groups'].isin(RARE_PATHOGENS), 'Sample group'] = 'Rare pathogens'

In [22]:
sc_labels['Pathogen groups'] = sc_labels['Pathogen groups'].replace({
    'Pseudomonas aeruginosa': 'Pseudomonas',
    'Pseudomonas aeruginosa; SARS-CoV-2': 'SARS-CoV-2; Pseudomonas',
    'Gram-*': 'Other Gram–',
    'Gram-*; Gram+': 'Other Gram–; Gram+'
})

In [23]:
sc_labels.loc[sc_labels['Cohort'].eq('LongCOVID'), 'Sample group'] = 'PASC'
sc_labels.loc[sc_labels['Pathogen groups'].eq('Healthy'), 'Sample group'] = 'Healthy'
sc_labels.loc[sc_labels['Pathogen groups'].eq('NPC'), 'Sample group'] = 'NPC'
sc_labels.loc[sc_labels['Pathogen groups'].isin(['Early SARS-CoV-2', 'Late SARS-CoV-2']), 'Sample group'] = 'Viral'
sc_labels.loc[sc_labels['Pathogen groups'].isin(
    ['Pseudomonas', 'Gram+', 'Other Gram–', 'Other Gram–; Gram+']
), 'Sample group'] = 'Bacterial'
sc_labels.loc[sc_labels['Pathogen groups'].isin(
    ['SARS-CoV-2; Pseudomonas', 'Early SARS-CoV-2; Gram+', 'Late SARS-CoV-2; Gram+']
), 'Sample group'] = 'Mixed'
sc_labels.loc[
    (
        sc_labels['pathogen_groups'].eq('pathogen-negative')
        & sc_labels['Sample group'].ne('Pathogen-negative pneumonia')
        & sc_labels['Pathogen groups'].eq('discard')
        & sc_labels['Pneumonia episode type'].isin(['CAP', 'HAP', 'VAP', 'VVAP'])
    ),
    'Sample group'
] = 'Pathogen cleared'
sc_labels['Sample group'] = sc_labels['Sample group'].fillna('Other')

In [24]:
sc_labels['Sample group'].value_counts(dropna=False)

Viral                          60
Bacterial                      58
Mixed                          33
Pathogen-negative pneumonia    32
Other                          28
NPC                            26
PASC                           25
Rare pathogens                 22
Healthy                         9
Pathogen cleared                8
Name: Sample group, dtype: int64

In [25]:
sc_labels['Pathogen groups'] = sc_labels['Pathogen groups'].replace({'discard': np.nan})

Is it safe to log the days?

In [26]:
sc_labels['Day of hospitalization (log)'].min()

1.0

In [27]:
sc_labels['Days on ventilator (log)'].min()

1.0

Yes!

In [28]:
sc_labels['Day of hospitalization (log)'] = np.log2(sc_labels['Day of hospitalization (log)'])
sc_labels['Days on ventilator (log)'] = np.log2(sc_labels['Days on ventilator (log)'])

In [29]:
sc_labels.drop(columns=['pathogen_groups', 'Pathogen negative', 'Cohort'], inplace=True)

In [30]:
sc_labels.shape

(301, 9)

In [31]:
ds_small.obs.individual.nunique()

301

In [32]:
ds_small.obs = ds_small.obs.merge(sc_labels, left_on='individual', right_index=True).loc[ds_small.obs_names]

Absolutely random library IDs

In [33]:
rng = np.random.default_rng(47656254)

In [34]:
lib_ids = rng.choice(range(1000, 2000), size=ds_small.obs.library_id.nunique(), replace=False)

In [35]:
anon_lib_ids = (
    'L-'
    + pd.Series(rng.choice(['J', 'Q', 'U', 'Y', 'F'], size=len(lib_ids)))
    + pd.Series(lib_ids).astype(str).str[1:]
)
anon_lib_ids

0      L-Q004
1      L-J967
2      L-Y721
3      L-F846
4      L-J515
        ...  
332    L-Q799
333    L-U907
334    L-U768
335    L-J480
336    L-U539
Length: 337, dtype: object

In [36]:
ds_small.obs.library_id = ds_small.obs.library_id.cat.rename_categories(list(anon_lib_ids))

Change cell IDs

In [37]:
ds_small.obs_names = ds_small.obs.library_id.astype(str) + '-' + ds_small.obs_names.str.split('_').str[1]

In [38]:
ds_small.obs.drop(columns='library_id', inplace=True)

Use anonymized patients and samples

In [39]:
anon_ids = pd.read_csv(common_data.ANON_IDS).dropna().set_index('bal_barcode').Anon_sample_id.to_dict()

In [40]:
ds_small.obs.individual = ds_small.obs.individual.replace(anon_ids)

In [41]:
ds_small.obs.rename(columns={
    'individual': 'Sample',
    'n_genes_by_counts': 'Number of genes',
    'total_counts': 'Number of UMIs',
    'pct_counts_mito': '% of mitochondrial reads',
    'pct_counts_ribo': '% of ribosomal reads',
    'Level_6': 'Cell type',
    'Sex_genes': 'Sex',
}, inplace=True)

In [42]:
ds_small.obs.Mortality = ds_small.obs.Mortality.replace({1: 'Died', 0: 'Survived'})

In [43]:
ds_small.obs.Immunocompromised = ds_small.obs.Immunocompromised.replace({1: 'Yes', 0: 'No'})

In [44]:
ds_small.obs['Patient'] = ds_small.obs.Sample.str.split('_').str[0]

In [45]:
ORDER = [
    'Patient',
    'Sex',
    'Immunocompromised',
    'Discharge disposition',
    'Mortality',

    'Sample',
    'Sample group',
    'Pathogen groups',
    'Pneumonia episode type',
    'SOFA score',
    'Days on ventilator (log)',
    'Day of hospitalization (log)',

    'Number of genes',
    'Number of UMIs',
    '% of mitochondrial reads',
    '% of ribosomal reads',
    'Cell type',
]

In [46]:
ds_small.obs = ds_small.obs[ORDER]

Vaguely remember that saving categories with NAs didn't work, so convert NAs to strings

In [47]:
for c in ds_small.obs.columns:
    if not pd.api.types.is_numeric_dtype(ds_small.obs[c]):
        ds_small.obs[c] = ds_small.obs[c].fillna('NA').astype('category')

Vaguely remember that `gene_ids` field for genes was causing strange behaviour in cellbrowser

In [48]:
ds_small.var.drop(columns='gene_ids', inplace=True)

Update cell types to paper figures

In [49]:
ds_small.obs['Cell type'] = ds_small.obs['Cell type'].cat.rename_categories(common_plots.CELL_TYPES_DISPLAY)
ds_small.obs['Cell type'] = ds_small.obs['Cell type'].cat.rename_categories({'Perivascular macrophages': 'Interstitial macrophages'})

In [ ]:
ds_small.write_h5ad(common_data._sc_root / '14_preprint_export/01_downsampled.h5ad')

In [ ]:
ds_small.obs.to_csv(common_data._sc_root / '14_preprint_export/01_downsampled-metadata.csv')

And in markers file too

In [ ]:
markers = pd.read_csv(common_data._sc_root / '09_final_full-1/09_final_full-1-markers.csv', index_col=0)
markers

,p_val,avg_logFC,pct.1,pct.2,p_val_adj,cluster,gene
0,0.0,14.602209,0.952576,0.001575,0.0,AT1 and AT2,SFTPB
10,0.0,13.722928,0.829918,0.000346,0.0,AT1 and AT2,SFTA2
1,0.0,10.498070,0.914813,0.007215,0.0,AT1 and AT2,KRT7
12,0.0,10.381162,0.832845,0.003761,0.0,AT1 and AT2,CEACAM6
19,0.0,9.854723,0.758489,0.003450,0.0,AT1 and AT2,GPRC5A
...,...,...,...,...,...,...,...
1364,0.0,1.585608,0.989603,0.749320,0.0,pDC,HLA-DRA
1390,0.0,1.320557,0.966379,0.846281,0.0,pDC,NACA
1396,0.0,1.132448,0.974075,0.870283,0.0,pDC,RACK1
1387,0.0,1.035872,0.999190,0.987593,0.0,pDC,MALAT1


In [53]:
markers.cluster = markers.cluster.replace(common_plots.CELL_TYPES_DISPLAY).replace({'Perivascular macrophages': 'Interstitial macrophages'})

In [ ]:
markers.to_csv(common_data._sc_root / '14_preprint_export/01_downsampled-markers.csv')